# The Ontology Spec and Units
This example describes how BDF's canonical column ontology is loaded and queried through the `bdf.spec` module. It is aimed at those who want to inspect the quantities BDF understands, pin a specific ontology version, or perform unit conversions with the spec as the single source of truth.

In [ ]:
from bdf import spec
from bdf.spec import COLUMN_ONTOLOGY, ColumnOntology, Quantity

## The default ontology
`bdf.spec.COLUMN_ONTOLOGY` is a module-level singleton built at import time from the ontology snapshot bundled with the installed package. This is the latest packaged version and is what every reader, normalizer, and validator uses by default. Iterating it yields `(mr_name, Quantity)` pairs.

In [ ]:
print("ontology version:", COLUMN_ONTOLOGY.ontology_version)
print("quantity count:", len(list(COLUMN_ONTOLOGY)))
print("first 8 mr_names:", [mr for mr, _ in COLUMN_ONTOLOGY][:8])

Each quantity is a `Quantity` model carrying the canonical unit, the human-readable label template, the machine-readable name, the ontology IRI, accepted synonyms, and documentation metadata. Look one up by `mr_name` with item access (or attribute access).

In [ ]:
q = COLUMN_ONTOLOGY["discharging_capacity_ah"]
print("mr_name:        ", q.mr_name)
print("unit:           ", q.unit)
print("label_template: ", q.label_template)
print("formatted_label:", q.formatted_label)
print("notation:       ", q.effective_notation)
print("iri:            ", q.iri)
print("required:       ", q.required)
print("synonyms:       ", q.synonyms[:5])
print("definition:     ", q.definition)

The required and optional label sets are derived directly from the loaded ontology, so they always reflect whichever version is active.

In [ ]:
print("required labels:", COLUMN_ONTOLOGY.required_labels())
print("optional labels:", COLUMN_ONTOLOGY.optional_labels()[:6], "...")

## Loading other versions or a custom ontology
The default singleton is fine for everyday use, but you can build an independent `ColumnOntology` and point it at a different source. Construct a fresh instance with `ColumnOntology.build()` so you never mutate the shared `COLUMN_ONTOLOGY` singleton.

`load_ttl(path)` adopts the quantities from any local Turtle file, which is the way to use a custom or in-development ontology. Here we load the snapshot bundled with the package to show the mechanism offline.

In [ ]:
import importlib.resources
from pathlib import Path

custom = ColumnOntology.build()
ttl_path = Path(str(importlib.resources.files("bdf.data").joinpath("bdf-ontology-snapshot.ttl")))
custom.load_ttl(ttl_path)
print("custom ontology version:", custom.ontology_version, "with", len(list(custom)), "quantities")

`load_latest(refresh=True)` fetches the live ontology from `w3id.org`, parses it, and caches it locally. Use it to pick up a release newer than the bundled snapshot. (This cell requires network access.)

In [ ]:
latest = ColumnOntology.build()
try:
    latest.load_latest(refresh=True)
    print("live ontology version:", latest.ontology_version)
except Exception as exc:
    print("live fetch unavailable, keeping bundled snapshot:", type(exc).__name__)

You can pin a specific release explicitly with `load_version`. It serves the cached copy when present, and otherwise fetches that release tag, verifies its `owl:versionInfo` matches, and caches it for next time.

In [ ]:
pinned = ColumnOntology.build()
pinned.load_version("1.1.0")
print("loaded pinned version:", pinned.ontology_version)

## Resolving labels to machine-readable names
Given a column label as it appears in a BDF dataframe, the ontology resolves it back to its canonical quantity. `mr_name_from_label` returns the machine-readable name; `quantity_from_label` returns the matching `Quantity` together with the unit parsed from the label. Both prefer non-deprecated quantities when a base label is shared.

In [ ]:
print("mr_name:", COLUMN_ONTOLOGY.mr_name_from_label("Discharging Capacity / Ah"))

matched, unit = COLUMN_ONTOLOGY.quantity_from_label("Voltage / mV")
print("matched quantity:", matched.mr_name)
print("unit from label: ", unit)

Two module-level helpers parse labels without touching the ontology: `parse_label` splits a label into its base and normalized unit, and `unit_from_label` returns just the unit.

In [ ]:
print("parse_label:    ", spec.parse_label("Voltage / mV"))
print("unit_from_label:", spec.unit_from_label("Test Time / s"))

## Unit conversion
Unit conversion is driven by the canonical unit recorded on each quantity. `get_unit_conversion(src, dst)` returns the `(scale, offset)` pair that maps a value in `src` units to `dst` units, or `None` when the units are dimensionally incompatible. The scale/offset form covers affine conversions such as °C to K.

In [ ]:
print("s  -> h:   ", spec.get_unit_conversion("s", "h"))
print("Ah -> mAh: ", spec.get_unit_conversion("Ah", "mAh"))
print("degC -> K: ", spec.get_unit_conversion("degC", "K"))
print("V  -> A:   ", spec.get_unit_conversion("V", "A"))

A `Quantity` knows its own canonical unit, so it can build conversions relative to itself. `convert_to(dst)` converts from the quantity's unit to `dst`, and `convert_from(src)` converts a source unit into the quantity's unit — exactly what a reader does when ingesting vendor data.

In [ ]:
cap = COLUMN_ONTOLOGY["discharging_capacity_ah"]
print("canonical unit:        ", cap.unit)
print("convert_to('mAh'):   ", cap.convert_to("mAh"))
print("convert_from('mAh'):   ", cap.convert_from("mAh"))

Applying a conversion is just `value * scale + offset`. The example below converts a millivolt reading into the canonical volt unit using the pair returned for the voltage quantity.

In [ ]:
voltage = COLUMN_ONTOLOGY["voltage_volt"]
scale, offset = voltage.convert_from("mV")
raw_mv = [3200.0, 3300.0, 3400.0]
converted_v = [v * scale + offset for v in raw_mv]
print("mV:", raw_mv)
print("V: ", converted_v)

## Published unit annotations
`unit` is a *normalized* string, chosen so that pint can parse it and conversions work: `Ah`, `degC`, `ohm`. The ontology publishes something slightly different — `schema:unitCode` (a UCUM code) and `schema:unitText` (a human-readable name). `Quantity` carries both verbatim as `unit_code` and `unit_text`.

The distinction matters when you emit rather than compute. Normalization is a BDF-local convention, so published output should use the annotations as the ontology states them, while conversion keeps using `unit`.

In [ ]:
cap = COLUMN_ONTOLOGY["discharging_capacity_ah"]
print("unit (normalized):", cap.unit)
print("unit_code:        ", cap.unit_code)
print("unit_text:        ", cap.unit_text)

For most terms the two agree. They diverge for exactly the codes pint cannot read as written, which is the whole reason the normalized form exists:

In [ ]:
diverging = sorted({(q.unit_code, q.unit) for _, q in COLUMN_ONTOLOGY if q.unit_code and q.unit_code != q.unit})
print("published code -> normalized unit:")
for published, normalized in diverging:
    print(f"  {published:>5} -> {normalized}")

A term with no unit at all — a label or a free-text identifier — has `unit is None` and empty strings for both annotations, so the two cases stay distinguishable.

In [ ]:
step_type = COLUMN_ONTOLOGY["step_type"]
print("unit:     ", step_type.unit)
print("unit_code:", repr(step_type.unit_code))
print("unit_text:", repr(step_type.unit_text))

Each column term also carries an EMMO measurement-unit restriction, and BDF deliberately does not read it. At the pinned BattINFO version the classes it names for capacity and energy — `AmpereHour` and `WattHour` — do not exist, so following the restriction would dead-end on the terms that most need a unit. The `schema:` annotations are the trustworthy source.

In [ ]:
import importlib.resources

from rdflib import URIRef
from rdflib.namespace import OWL, RDF, RDFS, SKOS

from bdf import vocabulary

snapshot = spec._graph_from_bytes(
    importlib.resources.files("bdf.data").joinpath("bdf-ontology-snapshot.ttl").read_bytes(),
    format="turtle",
)

restricted = sorted(
    {str(o).rsplit("#", 1)[-1] for o in snapshot.objects(None, OWL.someValuesFrom) if "emmo#" in str(o)}
)
for name in restricted:
    if not name.startswith("EMMO_"):  # opaque IRIs carry no readable key to look up
        print(f"  emmo:{name:<15} resolves: {vocabulary.has_term(name)}")

## What counts as a data column
`COLUMN_ONTOLOGY` holds BDF *data columns*, not every class in the ontology file. The discriminator is `rdfs:subClassOf sosa:ObservableProperty`: a class carrying it is something a dataset can have a column of, and `ColumnOntology.from_graph` admits nothing else.

This is not a formality. The snapshot ships with its EMMO imports unresolved; resolve them and the closure contributes thousands of labelled classes, none of which is a column. Filtering on a published parent keeps that flood out without a hand-maintained list.

In [ ]:
OBSERVABLE_PROPERTY = URIRef("http://www.w3.org/ns/sosa/ObservableProperty")

classes = set(snapshot.subjects(RDF.type, OWL.Class))
columns = {c for c in classes if (c, RDFS.subClassOf, OBSERVABLE_PROPERTY) in snapshot}
labelled = {c for c in classes if any(snapshot.objects(c, SKOS.prefLabel))}

print("owl:Class in the snapshot:      ", len(classes))
print("with a skos:prefLabel:          ", len(labelled))
print("subClassOf ObservableProperty:  ", len(columns))
print("admitted to COLUMN_ONTOLOGY:    ", len(list(COLUMN_ONTOLOGY)))

The classes left out are the EMMO unit classes the snapshot references, plus `sosa:ObservableProperty` itself. None of them is labelled, and none is a quantity anyone would put in a column:

In [ ]:
def local_name(iri):
    return str(iri).rsplit("#", 1)[-1].rsplit("/", 1)[-1]


for name in sorted(local_name(c) for c in classes - columns):
    print(" ", name)

Excluded does not mean unreachable. A class that is not a column is *metadata vocabulary*, and `bdf.vocabulary` resolves it by key against the pinned BattINFO context. The two lookups answer different questions — one asks what a column means, the other what a term is called.

In [ ]:
print("Volt in COLUMN_ONTOLOGY: ", "Volt" in COLUMN_ONTOLOGY)
print("vocabulary.resolve:      ", vocabulary.resolve("Volt"))

volts = COLUMN_ONTOLOGY["voltage_volt"]
print("the voltage column:      ", volts.iri)